# Beginner E4: combine classical and Whisper accent scores

This notebook implements **E4 — Hybrid ensemble** from `docs/Singlish_draft.md`. E1 supplies acoustic features, E2 supplies frozen Whisper embeddings, and weighted soft voting combines their Singapore-class scores:

`P(final) = w × P(E1) + (1 − w) × P(E2)`

We choose the E1 model and weight using **validation speakers only**, then compare E1, E2 and E4 on the same unseen test speakers. Label `1` means Singapore speech; label `0` means non-Singapore speech. These are model scores for an accent variety, not judgments about ethnicity, nationality or identity.

## Before you start

Run one E1 beginner notebook through its **Save the experiment** cell, then run `notebooks/E2_01_whisper_classifier_beginner.ipynb` through **Extract once, then cache**. Use the same `Singlish/data/audio` folder and `MAX_CLIPS_PER_CLASS` setting. E4 reads the cached features and embeddings and trains fresh classifiers on its own shared split. It does not reuse E1/E2 test metrics or their saved fitted models, because those notebooks may have chosen different test speakers.

This notebook is written for Colab and Google Drive, like E3. Set `USE_GOOGLE_DRIVE = False` and edit `PROJECT_ROOT` to run locally. No GPU is needed once E2 embeddings have been cached.

In [ ]:
# Run once in Colab.
%pip -q install numpy scikit-learn matplotlib joblib

## 1. Point to the saved E1 and E2 caches

E1's SVM and Logistic Regression notebooks use `beginner_acoustic_features.npz`; Random Forest uses `beginner_random_forest_features.npz`. Either contains the same 39 feature definitions. If both are present, E4 uses the first one and checks that its clip list matches E2. Edit the cache path below if you prefer the other E1 cache.

In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/Singlish')
MAX_CLIPS_PER_CLASS = 100  # Match E2; None means all cached audio.
RANDOM_SEED = 42

if MAX_CLIPS_PER_CLASS is not None and (
    not isinstance(MAX_CLIPS_PER_CLASS, int)
    or isinstance(MAX_CLIPS_PER_CLASS, bool)
    or MAX_CLIPS_PER_CLASS < 1
):
    raise ValueError('MAX_CLIPS_PER_CLASS must be None or a positive integer.')

E1_DIR = PROJECT_ROOT / 'outputs' / 'e1'
E2_DIR = PROJECT_ROOT / 'outputs' / 'e2'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'e4'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
E1_CACHE = E1_DIR / 'beginner_acoustic_features.npz'
if not E1_CACHE.is_file():
    E1_CACHE = E1_DIR / 'beginner_random_forest_features.npz'
cache_suffix = '' if MAX_CLIPS_PER_CLASS is None else f'_max{MAX_CLIPS_PER_CLASS}'
E2_CACHE = E2_DIR / f'beginner_whisper_embeddings{cache_suffix}.npz'
print('E1 cache:', E1_CACHE)
print('E2 cache:', E2_CACHE)
print('E4 output:', OUTPUT_DIR)

## 2. Match the same recordings in both branches

One row must refer to the same clip, class label and speaker in both caches. E1 or E2 may have skipped a file during feature extraction. We keep only the intersection, report how many clips were dropped, and reject duplicate paths or conflicting metadata. If the intersection is unexpectedly small, check the E1/E2 input folders before continuing.

In [ ]:
import csv
import json
from collections import Counter

import joblib
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (balanced_accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


def read_cache(path, feature_key):
    if not path.is_file():
        raise FileNotFoundError(f'Missing {path}. Run the matching E1 or E2 notebook first.')
    with np.load(path, allow_pickle=False) as cache:
        required = {feature_key, 'labels', 'groups', 'paths'}
        missing = required - set(cache.files)
        if missing:
            raise ValueError(f'{path} is missing {sorted(missing)}')
        features = cache[feature_key].astype(np.float32)
        labels = cache['labels'].astype(np.int64)
        groups = cache['groups'].astype(str)
        paths = cache['paths'].astype(str)
        model_name = str(cache['model_name']) if 'model_name' in cache.files else None
    if features.ndim != 2 or not (len(features) == len(labels) == len(groups) == len(paths)):
        raise ValueError(f'Inconsistent array shapes in {path}')
    if len(set(paths)) != len(paths):
        raise ValueError(f'Duplicate clip paths in {path}')
    if not np.isfinite(features).all() or not set(np.unique(labels)).issubset({0, 1}):
        raise ValueError(f'Invalid features or labels in {path}')
    return features, labels, groups, paths, model_name


e1_X, e1_y, e1_groups, e1_paths, _ = read_cache(E1_CACHE, 'features')
e2_X, e2_y, e2_groups, e2_paths, whisper_model = read_cache(E2_CACHE, 'embeddings')
if e1_X.shape[1] != 39:
    raise ValueError('Expected the 39 acoustic features from an E1 beginner notebook.')

e1_lookup = {path: i for i, path in enumerate(e1_paths)}
e2_lookup = {path: i for i, path in enumerate(e2_paths)}
shared_paths = sorted(e1_lookup.keys() & e2_lookup.keys())
if not shared_paths:
    raise ValueError('No shared clip paths. Check that E1 and E2 used the same audio folder.')
e1_rows = np.array([e1_lookup[path] for path in shared_paths])
e2_rows = np.array([e2_lookup[path] for path in shared_paths])
if not np.array_equal(e1_y[e1_rows], e2_y[e2_rows]):
    raise ValueError('E1 and E2 labels disagree for a shared clip.')
if not np.array_equal(e1_groups[e1_rows], e2_groups[e2_rows]):
    raise ValueError('E1 and E2 speaker IDs disagree for a shared clip.')

X_ml, X_deep = e1_X[e1_rows], e2_X[e2_rows]
y, groups, paths = e1_y[e1_rows], e1_groups[e1_rows], np.asarray(shared_paths)
print('Shared clips:', len(paths), '| labels:', Counter(y))
print('Only in E1:', len(e1_paths) - len(paths), '| only in E2:', len(e2_paths) - len(paths))
print('Unique speakers:', len(set(groups)), '| Whisper:', whisper_model)

## 3. Split by speaker before fitting anything

Each class's speakers are shuffled independently. About 70% go to training, 15% to validation and 15% to test. All clips from one speaker stay together. We need at least seven speakers per class so that each partition can contain both labels. The test speakers stay untouched until model and weight selection are complete.

In [ ]:
def make_speaker_split(labels, speaker_ids, seed):
    rng = np.random.default_rng(seed)
    speaker_partition = {}
    for label in (0, 1):
        speakers = np.unique(speaker_ids[labels == label])
        if len(speakers) < 7:
            raise ValueError(f'Class {label} needs at least seven speakers; found {len(speakers)}.')
        rng.shuffle(speakers)
        n_test = max(1, round(0.15 * len(speakers)))
        n_val = max(1, round(0.15 * len(speakers)))
        for speaker in speakers[:n_test]:
            speaker_partition[speaker] = 'test'
        for speaker in speakers[n_test:n_test + n_val]:
            speaker_partition[speaker] = 'validation'
        for speaker in speakers[n_test + n_val:]:
            speaker_partition[speaker] = 'train'
    split = {
        name: np.flatnonzero(np.array([speaker_partition[g] == name for g in speaker_ids]))
        for name in ('train', 'validation', 'test')
    }
    for name, indices in split.items():
        if set(np.unique(labels[indices])) != {0, 1}:
            raise ValueError(f'{name} is missing a class.')
        print(name, '| clips:', len(indices), '| speakers:', len(set(speaker_ids[indices])),
              '| labels:', Counter(labels[indices]))
    speaker_sets = [set(speaker_ids[split[name]]) for name in split]
    assert all(not (a & b) for i, a in enumerate(speaker_sets) for b in speaker_sets[i + 1:])
    return split

split = make_speaker_split(y, groups, RANDOM_SEED)
train_idx, val_idx, test_idx = (split[name] for name in ('train', 'validation', 'test'))

## 4. Choose the classical model on validation speakers

The proposal calls for the best classical E1 model. We train SVM, Random Forest and Logistic Regression on the *same* acoustic training rows. Their validation macro F1 decides the winner; balanced accuracy breaks ties. We then fit a fresh linear head on the E2 Whisper embeddings using the same training clips. Scaling is fitted inside the relevant training pipelines.

In [ ]:
e1_candidates = {
    'SVM': Pipeline([
        ('scale', StandardScaler()),
        ('classifier', SVC(kernel='rbf', C=1.0, gamma='scale',
                           class_weight='balanced', probability=True,
                           random_state=RANDOM_SEED)),
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, max_features='sqrt', min_samples_leaf=2,
        class_weight='balanced', n_jobs=-1, random_state=RANDOM_SEED,
    ),
    'Logistic Regression': Pipeline([
        ('scale', StandardScaler()),
        ('classifier', LogisticRegression(C=1.0, class_weight='balanced',
                                          max_iter=2000, random_state=RANDOM_SEED)),
    ]),
}


def singapore_probability(model, features):
    classes = list(model.classes_)
    if classes != [0, 1]:
        raise ValueError(f'Expected class order [0, 1], got {classes}')
    scores = model.predict_proba(features)[:, classes.index(1)]
    if not np.isfinite(scores).all() or np.any((scores < 0) | (scores > 1)):
        raise ValueError('Invalid probability score.')
    return scores


def selection_scores(truth, probabilities):
    predicted = (probabilities >= 0.5).astype(int)
    return (float(f1_score(truth, predicted, average='macro')),
            float(balanced_accuracy_score(truth, predicted)))


candidate_results = {}
for name, candidate in e1_candidates.items():
    candidate.fit(X_ml[train_idx], y[train_idx])
    val_prob = singapore_probability(candidate, X_ml[val_idx])
    candidate_results[name] = {
        'model': candidate, 'validation': selection_scores(y[val_idx], val_prob)
    }
    print(f'{name}: validation macro F1={candidate_results[name]["validation"][0]:.3f}, '
          f'balanced accuracy={candidate_results[name]["validation"][1]:.3f}')

best_e1_name = max(candidate_results, key=lambda name: candidate_results[name]['validation'])
ml_model = candidate_results[best_e1_name]['model']
deep_model = Pipeline([
    ('scale', StandardScaler()),
    ('head', LogisticRegression(C=1.0, class_weight='balanced',
                                max_iter=2000, random_state=RANDOM_SEED)),
])
deep_model.fit(X_deep[train_idx], y[train_idx])
print('Selected E1 model:', best_e1_name)
print('E2 validation scores:', selection_scores(
    y[val_idx], singapore_probability(deep_model, X_deep[val_idx])))

## 5. Choose the voting weight on validation speakers

For each weight below, we average the **Singapore-class** scores. `w=1` is E1 alone; `w=0` is E2 alone. We select the largest validation macro F1, then balanced accuracy; ties prefer a 50/50 blend. Endpoints are included so the result can honestly say when blending did not help. The threshold stays at 0.5.

In [ ]:
p_ml_val = singapore_probability(ml_model, X_ml[val_idx])
p_deep_val = singapore_probability(deep_model, X_deep[val_idx])
weights = np.linspace(0, 1, 11)
weight_rows = []
for weight in weights:
    combined = weight * p_ml_val + (1 - weight) * p_deep_val
    macro_f1, balanced_acc = selection_scores(y[val_idx], combined)
    weight_rows.append({'weight_e1': float(weight), 'macro_f1': macro_f1,
                        'balanced_accuracy': balanced_acc})
    print(f'E1 weight={weight:.1f}: macro F1={macro_f1:.3f}, balanced accuracy={balanced_acc:.3f}')

best_weight_row = max(weight_rows, key=lambda row: (
    row['macro_f1'], row['balanced_accuracy'], -abs(row['weight_e1'] - 0.5)
))
w_ml = best_weight_row['weight_e1']
print(f'Selected weight: E1={w_ml:.1f}, E2={1 - w_ml:.1f}')

## 6. Compare all three on unseen test speakers

Now we score the test set once. Rows in each confusion matrix are true labels; columns are predicted labels, ordered `[0, 1]`. Macro F1 gives both classes equal weight, and balanced accuracy averages class recall. The detailed report includes precision, recall and F1 for each class. Since these are model probability scores, do not assume they are well calibrated without a separate calibration check.

In [ ]:
p_ml_test = singapore_probability(ml_model, X_ml[test_idx])
p_deep_test = singapore_probability(deep_model, X_deep[test_idx])
p_final_test = w_ml * p_ml_test + (1 - w_ml) * p_deep_test


def evaluate(truth, probabilities):
    predicted = (probabilities >= 0.5).astype(int)
    return {
        'macro_f1': float(f1_score(truth, predicted, average='macro')),
        'balanced_accuracy': float(balanced_accuracy_score(truth, predicted)),
        'confusion_matrix': confusion_matrix(truth, predicted, labels=[0, 1]).tolist(),
        'per_class': classification_report(
            truth, predicted, labels=[0, 1],
            target_names=['Non-Singapore', 'Singapore'],
            output_dict=True, zero_division=0,
        ),
    }

results = {
    f'E1 ({best_e1_name})': evaluate(y[test_idx], p_ml_test),
    'E2 (Whisper)': evaluate(y[test_idx], p_deep_test),
    'E4 (weighted vote)': evaluate(y[test_idx], p_final_test),
}
for name, result in results.items():
    print(f'{name}: macro F1={result["macro_f1"]:.3f}, '
          f'balanced accuracy={result["balanced_accuracy"]:.3f}')
    for class_name in ('Non-Singapore', 'Singapore'):
        scores = result['per_class'][class_name]
        print(f'  {class_name}: precision={scores["precision"]:.3f}, '
              f'recall={scores["recall"]:.3f}, F1={scores["f1-score"]:.3f}')

best_single_f1 = max(results[f'E1 ({best_e1_name})']['macro_f1'],
                     results['E2 (Whisper)']['macro_f1'])
print('E4 minus best single-model test macro F1:',
      round(results['E4 (weighted vote)']['macro_f1'] - best_single_f1, 3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)
for ax, (name, result) in zip(axes, results.items()):
    matrix = np.asarray(result['confusion_matrix'])
    ax.imshow(matrix, cmap='Blues', vmin=0, vmax=max(1, max(
        np.asarray(item['confusion_matrix']).max() for item in results.values())))
    for (row, col), value in np.ndenumerate(matrix):
        ax.text(col, row, str(value), ha='center', va='center')
    ax.set_title(name)
    ax.set_xticks([0, 1], ['Non-SG', 'SG'])
    ax.set_yticks([0, 1], ['Non-SG', 'SG'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
confusion_path = OUTPUT_DIR / 'beginner_e4_confusion_matrices.png'
fig.savefig(confusion_path, dpi=160)
plt.show()
print('Saved:', confusion_path)

## 7. Save the result and inspect a cached clip

The CSV keeps each held-out clip's E1, E2 and final Singapore scores. The JSON records settings, splits, selection results and test metrics. The two saved classifiers operate on cached 39-value E1 features and E2 embeddings; a new audio file would need the same E1 preprocessing and frozen Whisper encoder first.

In [ ]:
prediction_path = OUTPUT_DIR / 'beginner_e4_test_predictions.csv'
with prediction_path.open('w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['path', 'speaker_id', 'true_label', 'e1_singapore_score',
                     'e2_singapore_score', 'final_singapore_score', 'predicted_label'])
    for row, ml, deep, final in zip(test_idx, p_ml_test, p_deep_test, p_final_test):
        writer.writerow([paths[row], groups[row], int(y[row]), float(ml),
                         float(deep), float(final), int(final >= 0.5)])

split_path = OUTPUT_DIR / 'beginner_e4_split.npz'
np.savez_compressed(split_path, paths=paths, labels=y, groups=groups,
                    train_indices=train_idx, validation_indices=val_idx,
                    test_indices=test_idx)
model_path = OUTPUT_DIR / 'beginner_e4_models.joblib'
joblib.dump({'e1_model': ml_model, 'e2_head': deep_model,
             'e1_weight': w_ml, 'e1_name': best_e1_name,
             'whisper_model': whisper_model}, model_path)
report = {
    'e1_cache': str(E1_CACHE), 'e2_cache': str(E2_CACHE),
    'whisper_model': whisper_model,
    'label_meaning': {'0': 'non_singapore', '1': 'singapore'},
    'shared_clips': int(len(paths)),
    'e1_only_clips': int(len(e1_paths) - len(paths)),
    'e2_only_clips': int(len(e2_paths) - len(paths)),
    'split': {name: {'clips': int(len(indices)),
                     'speakers': int(len(set(groups[indices]))),
                     'label_counts': {str(label): int(np.sum(y[indices] == label))
                                      for label in (0, 1)}}
              for name, indices in split.items()},
    'e1_validation': {name: {'macro_f1': scores['validation'][0],
                             'balanced_accuracy': scores['validation'][1]}
                      for name, scores in candidate_results.items()},
    'selected_e1': best_e1_name,
    'weight_validation': weight_rows,
    'selected_e1_weight': w_ml,
    'test_metrics': results,
    'caution': 'Corpus and recording differences may be learned instead of accent; scores are not calibrated identity judgments.',
}
report_path = OUTPUT_DIR / 'beginner_e4_report.json'
report_path.write_text(json.dumps(report, indent=2) + '\n')
for path in (prediction_path, split_path, model_path, report_path, confusion_path):
    print('Saved:', path)

In [ ]:
def score_cached_recording(audio_path):
    # Looks up a clip already represented in both caches.
    key = str(Path(audio_path))
    matches = np.flatnonzero(paths == key)
    if len(matches) != 1:
        raise ValueError('Clip is not in both caches; use a path printed in the predictions CSV.')
    row = int(matches[0])
    ml = float(singapore_probability(ml_model, X_ml[row:row + 1])[0])
    deep = float(singapore_probability(deep_model, X_deep[row:row + 1])[0])
    final = w_ml * ml + (1 - w_ml) * deep
    return {'e1_singapore_score': ml, 'e2_singapore_score': deep,
            'final_singapore_score': final, 'predicted_label': int(final >= 0.5)}

example_path = paths[test_idx[0]]
print('Example:', example_path)
print(score_cached_recording(example_path))

## What to report

1. The number of shared clips and disjoint speakers in each partition.
2. The validation-selected E1 model and E1/E2 weights.
3. Test macro F1, balanced accuracy, class precision/recall/F1 and all three confusion matrices.
4. Whether E4 improved over the stronger single branch on **test** speakers. If a weight of 0 or 1 wins, report that the validation set favored one branch rather than claiming a fusion gain.
5. Limits: the two corpora can differ in microphones, prompts and recording style; probability scores can be uncalibrated. This experiment identifies accent patterns in this dataset, not a person's background.